# Multinomial Logistic Regression: Error Analysis & Strategy Evaluation

## Brain Connectivity Classification: Why Multinomial Falls Short

---

### The Classification Challenge

This notebook analyzes the performance of multinomial logistic regression for brain region classification using functional connectivity fingerprints. We evaluate three classification strategies:

| Strategy | Regions | Scope |
|----------|---------|-------|
| **Full Connectivity** | 232 | All regions, both hemispheres |
| **Left Hemisphere** | 116 | Left hemisphere regions only |
| **Right Hemisphere** | 116 | Right hemisphere regions only |

---

### Why Separate Hemisphere Models?

The brain exhibits **bilateral symmetry** — most cortical and subcortical structures have homologous regions in both hemispheres. When classifying brain regions from connectivity fingerprints, a 232-region classifier must simultaneously solve two distinct problems:

1. **Inter-hemisphere discrimination**: Distinguishing left from right homologues (e.g., Left Visual Cortex vs. Right Visual Cortex)
2. **Intra-hemisphere discrimination**: Distinguishing different networks/regions within a hemisphere (e.g., Left Visual vs. Left Motor)

These are fundamentally different classification challenges:
- **Inter-hemisphere confusion** arises from the high similarity of homologous regions
- **Intra-hemisphere confusion** arises from functional overlap between networks

By training **separate models per hemisphere**, we:
- **Isolate** the intra-hemisphere classification problem
- **Eliminate** inter-hemisphere errors by design
- **Quantify** how much error is attributable to hemisphere confusion vs. network confusion

This decomposition reveals *where* the multinomial classifier struggles and *why* alternative approaches may be needed.

---

### Notebook Structure

1. **Setup & Data Loading**
2. **Strategy Overview**: Performance comparison across all three models
3. **Full Model (232 regions)**: Detailed error analysis
4. **Left Hemisphere Model (116 regions)**: Separate analysis
5. **Right Hemisphere Model (116 regions)**: Separate analysis
6. **Cross-Strategy Comparison**: What hemisphere separation reveals
7. **Limitations of Multinomial**: Unanswered questions
8. **Motivation for OvR and OvO**: Why we need different approaches

---
## 1. Setup & Data Loading

In [1]:
# Core Libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings

# Statistical Analysis
from scipy import stats
from sklearn.metrics import confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Path Configuration
PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

# Directory Setup
full_multi_dir = RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial'
LH_multi_dir = RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'multinomial'
RH_multi_dir = RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'multinomial'

full_task_dir = RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing'
LH_task_dir = RESULTS_DIR / 'hemisphere_analysis' / 'left_hemisphere' / 'task_testing_multinomial'
RH_task_dir = RESULTS_DIR / 'hemisphere_analysis' / 'right_hemisphere' / 'task_testing_multinomial'

print(f"Results Directory: {RESULTS_DIR}")

Results Directory: /home/sjoon/projects/brain_connectivity_classifier/data/results


In [3]:
# Helper Functions
def load_json(filepath):
    with open(filepath, 'r') as f:
        return json.load(f)

def load_npy(filepath):
    return np.load(filepath, allow_pickle=True)

def load_csv(filepath):
    return pd.read_csv(filepath)

def load_bundle(m_dir, t_dir):
    """Load standard file patterns from directories."""
    return [
        load_json(m_dir / 'overall_metrics.json'), 
        load_json(m_dir / 'fold_metrics.json'),
        load_csv(m_dir / 'network_metrics.csv'), 
        load_csv(m_dir / 'per_region_metrics.csv'),
        load_npy(m_dir / 'confusion_matrix.npy'), 
        load_npy(m_dir / 'cv_predictions.npy'),
        load_npy(m_dir / 'cv_probabilities.npy'), 
        load_npy(m_dir / 'cv_true_labels.npy'),
        load_npy(m_dir / 'cv_fold_indices.npy'), 
        load_json(t_dir / 'task_testing_summary.json'),
        load_csv(t_dir / 'task_network_metrics.csv'), 
        load_csv(t_dir / 'task_per_region_metrics.csv'),
        load_npy(t_dir / 'task_confusion_matrix.npy'), 
        load_npy(t_dir / 'task_predictions.npy'),
        load_npy(t_dir / 'task_probabilities.npy'), 
        load_npy(t_dir / 'task_true_labels.npy')
    ]

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [4]:
# Load All Data

# Full Connectivity Model (232 regions)
overall_metrics_full, fold_metrics_full, network_metrics_cv_full, per_region_metrics_cv_full, \
confusion_matrix_cv_full, cv_predictions_full, cv_probabilities_full, cv_true_labels_full, \
cv_fold_indices_full, task_summary_full, network_metrics_task_full, per_region_metrics_task_full, \
confusion_matrix_task_full, task_predictions_full, task_probabilities_full, task_true_labels_full = load_bundle(full_multi_dir, full_task_dir)

region_info_full = load_csv(full_multi_dir / 'region_info.csv')

# Left Hemisphere Model (116 regions)
overall_metrics_lh, fold_metrics_lh, network_metrics_cv_lh, per_region_metrics_cv_lh, \
confusion_matrix_cv_lh, cv_predictions_lh, cv_probabilities_lh, cv_true_labels_lh, \
cv_fold_indices_lh, task_summary_lh, network_metrics_task_lh, per_region_metrics_task_lh, \
confusion_matrix_task_lh, task_predictions_lh, task_probabilities_lh, task_true_labels_lh = load_bundle(LH_multi_dir, LH_task_dir)

region_info_lh = region_info_full[region_info_full['hemisphere'] == 'left'].copy()
region_info_lh['region_idx'] = range(len(region_info_lh))

# Right Hemisphere Model (116 regions)
overall_metrics_rh, fold_metrics_rh, network_metrics_cv_rh, per_region_metrics_cv_rh, \
confusion_matrix_cv_rh, cv_predictions_rh, cv_probabilities_rh, cv_true_labels_rh, \
cv_fold_indices_rh, task_summary_rh, network_metrics_task_rh, per_region_metrics_task_rh, \
confusion_matrix_task_rh, task_predictions_rh, task_probabilities_rh, task_true_labels_rh = load_bundle(RH_multi_dir, RH_task_dir)

region_info_rh = region_info_full[region_info_full['hemisphere'] == 'right'].copy()
region_info_rh['region_idx'] = range(len(region_info_rh))

print("✓ All data loaded successfully")

✓ All data loaded successfully


In [5]:
# Network Mapping (Yeo-7 + Subcortical)
network_to_major = {
    'VisCent': 'Visual', 'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor', 'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention', 'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention', 'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic', 'LimbicB': 'Limbic',
    'ContA': 'Control', 'ContB': 'Control', 'ContC': 'Control',
    'DefaultA': 'Default', 'DefaultB': 'Default', 'DefaultC': 'Default',
    'TempPar': 'Default',
    'Hippocampus_ant': 'Subcortical', 'Hippocampus_post': 'Subcortical',
    'Amygdala_lat': 'Subcortical', 'Amygdala_med': 'Subcortical',
    'Thalamus_DA': 'Subcortical', 'Thalamus_DP': 'Subcortical',
    'Thalamus_VA': 'Subcortical', 'Thalamus_VP': 'Subcortical',
    'Caudate_ant': 'Subcortical', 'Caudate_post': 'Subcortical',
    'Putamen_ant': 'Subcortical', 'Putamen_post': 'Subcortical',
    'Pallidum_ant': 'Subcortical', 'Pallidum_post': 'Subcortical',
    'Accumbens_core': 'Subcortical', 'Accumbens_shell': 'Subcortical'
}

for df in [region_info_full, region_info_lh, region_info_rh]:
    df['major_network'] = df['network'].map(network_to_major)

print("✓ Network mapping applied")

✓ Network mapping applied


---
## 2. Strategy Overview

Before diving into detailed analysis, we compare the overall performance of all three models.

In [6]:
# Performance Summary Table
strategy_summary = pd.DataFrame([
    {
        'Model': 'Full (232 regions)',
        'Hemisphere': 'Both',
        'N_Classes': 232,
        'CV_Accuracy': overall_metrics_full['accuracy'],
        'Task_Accuracy': task_summary_full['task_test_accuracy'],
        'Accuracy_Drop': task_summary_full['accuracy_drop'],
        'CV_Samples': len(cv_predictions_full),
        'Task_Samples': len(task_predictions_full)
    },
    {
        'Model': 'Left Hemisphere (116 regions)',
        'Hemisphere': 'Left',
        'N_Classes': 116,
        'CV_Accuracy': overall_metrics_lh['accuracy'],
        'Task_Accuracy': task_summary_lh['task_test_accuracy'],
        'Accuracy_Drop': task_summary_lh['accuracy_drop'],
        'CV_Samples': len(cv_predictions_lh),
        'Task_Samples': len(task_predictions_lh)
    },
    {
        'Model': 'Right Hemisphere (116 regions)',
        'Hemisphere': 'Right',
        'N_Classes': 116,
        'CV_Accuracy': overall_metrics_rh['accuracy'],
        'Task_Accuracy': task_summary_rh['task_test_accuracy'],
        'Accuracy_Drop': task_summary_rh['accuracy_drop'],
        'CV_Samples': len(cv_predictions_rh),
        'Task_Samples': len(task_predictions_rh)
    }
])

print("="*95)
print("MODEL PERFORMANCE OVERVIEW")
print("="*95)
print(strategy_summary.to_string(index=False))
print("="*95)

MODEL PERFORMANCE OVERVIEW
                         Model Hemisphere  N_Classes  CV_Accuracy  Task_Accuracy  Accuracy_Drop  CV_Samples  Task_Samples
            Full (232 regions)       Both        232     0.924126       0.892392       0.097813       51968         46400
 Left Hemisphere (116 regions)       Left        116     0.918180       0.869310       0.108022       25984         23200
Right Hemisphere (116 regions)      Right        116     0.913562       0.863060       0.114310       25984         23200


In [7]:
# Visualize performance comparison
fig = go.Figure()

models = ['Full (232)', 'Left (116)', 'Right (116)']
cv_accs = [overall_metrics_full['accuracy'], overall_metrics_lh['accuracy'], overall_metrics_rh['accuracy']]
task_accs = [task_summary_full['task_test_accuracy'], task_summary_lh['task_test_accuracy'], task_summary_rh['task_test_accuracy']]

fig.add_trace(go.Bar(name='Rest (CV)', x=models, y=[a*100 for a in cv_accs], 
                     marker_color='steelblue', text=[f'{a:.1%}' for a in cv_accs], textposition='outside'))
fig.add_trace(go.Bar(name='Task', x=models, y=[a*100 for a in task_accs], 
                     marker_color='coral', text=[f'{a:.1%}' for a in task_accs], textposition='outside'))

fig.update_layout(
    title='Classification Accuracy by Model',
    yaxis_title='Accuracy (%)',
    yaxis_range=[80, 100],
    barmode='group',
    template='plotly_white',
    height=450, width=700
)

fig.show()

---
## 3. Full Model Analysis (232 Regions)

The full model attempts to classify all 232 brain regions simultaneously. This analysis examines the types of errors it makes.

In [8]:
def analyze_errors_detailed(true_labels, pred_labels, region_info_df, model_name, condition):
    """Comprehensive error analysis with hemisphere and network breakdown."""
    err_mask = true_labels != pred_labels
    total_samples = len(true_labels)
    total_errors = err_mask.sum()
    
    if total_errors == 0:
        return {'model': model_name, 'condition': condition, 'total_errors': 0}
    
    lookup = region_info_df.set_index('region_idx')
    
    # Build error DataFrame
    df = pd.DataFrame({
        'true_idx': true_labels[err_mask],
        'pred_idx': pred_labels[err_mask]
    })
    
    df['true_network'] = df['true_idx'].map(lookup['network'])
    df['pred_network'] = df['pred_idx'].map(lookup['network'])
    df['true_major'] = df['true_idx'].map(lookup['major_network'])
    df['pred_major'] = df['pred_idx'].map(lookup['major_network'])
    df['true_hemi'] = df['true_idx'].map(lookup['hemisphere'])
    df['pred_hemi'] = df['pred_idx'].map(lookup['hemisphere'])
    
    # Error type classification
    same_hemi = df['true_hemi'] == df['pred_hemi']
    same_network = df['true_network'] == df['pred_network']
    same_major = df['true_major'] == df['pred_major']
    
    return {
        'model': model_name,
        'condition': condition,
        'total_samples': total_samples,
        'total_errors': total_errors,
        'error_rate': total_errors / total_samples,
        'within_hemi_network': (same_hemi & same_network).sum(),
        'within_hemi_diff_network': (same_hemi & ~same_network).sum(),
        'cross_hemi_same_network': (~same_hemi & same_network).sum(),
        'cross_hemi_diff_network': (~same_hemi & ~same_network).sum(),
        'within_major_network': same_major.sum(),
        'cross_major_network': (~same_major).sum(),
        'error_df': df
    }

# Analyze Full Model
full_rest_analysis = analyze_errors_detailed(cv_true_labels_full, cv_predictions_full, region_info_full, 'Full (232)', 'Rest')
full_task_analysis = analyze_errors_detailed(task_true_labels_full, task_predictions_full, region_info_full, 'Full (232)', 'Task')

In [9]:
# Display Full Model Error Breakdown
print("="*85)
print("FULL MODEL (232 REGIONS) - ERROR ANALYSIS")
print("="*85)

for analysis in [full_rest_analysis, full_task_analysis]:
    total = analysis['total_errors']
    print(f"\n{analysis['condition']} Condition:")
    print(f"  Total Samples: {analysis['total_samples']:,}")
    print(f"  Total Errors:  {total:,} ({analysis['error_rate']:.2%})")
    print(f"\n  Error Decomposition:")
    print(f"  ├─ Same Hemisphere, Same Network:      {analysis['within_hemi_network']:>5,} ({analysis['within_hemi_network']/total*100:>5.1f}%)")
    print(f"  ├─ Same Hemisphere, Different Network: {analysis['within_hemi_diff_network']:>5,} ({analysis['within_hemi_diff_network']/total*100:>5.1f}%)")
    print(f"  ├─ Different Hemisphere, Same Network: {analysis['cross_hemi_same_network']:>5,} ({analysis['cross_hemi_same_network']/total*100:>5.1f}%)")
    print(f"  └─ Different Hemisphere, Diff Network: {analysis['cross_hemi_diff_network']:>5,} ({analysis['cross_hemi_diff_network']/total*100:>5.1f}%)")
    
    cross_hemi_total = analysis['cross_hemi_same_network'] + analysis['cross_hemi_diff_network']
    print(f"\n  → Cross-Hemisphere Errors (Total):     {cross_hemi_total:>5,} ({cross_hemi_total/total*100:>5.1f}%)")

print("\n" + "="*85)

FULL MODEL (232 REGIONS) - ERROR ANALYSIS

Rest Condition:
  Total Samples: 51,968
  Total Errors:  3,943 (7.59%)

  Error Decomposition:
  ├─ Same Hemisphere, Same Network:        167 (  4.2%)
  ├─ Same Hemisphere, Different Network: 1,744 ( 44.2%)
  ├─ Different Hemisphere, Same Network:   284 (  7.2%)
  └─ Different Hemisphere, Diff Network: 1,748 ( 44.3%)

  → Cross-Hemisphere Errors (Total):     2,032 ( 51.5%)

Task Condition:
  Total Samples: 46,400
  Total Errors:  4,993 (10.76%)

  Error Decomposition:
  ├─ Same Hemisphere, Same Network:        244 (  4.9%)
  ├─ Same Hemisphere, Different Network: 2,292 ( 45.9%)
  ├─ Different Hemisphere, Same Network:   325 (  6.5%)
  └─ Different Hemisphere, Diff Network: 2,132 ( 42.7%)

  → Cross-Hemisphere Errors (Total):     2,457 ( 49.2%)



In [10]:
# Visualize Full Model Error Types
fig = make_subplots(rows=1, cols=2, subplot_titles=('Rest (CV)', 'Task'),
                    specs=[[{'type': 'pie'}, {'type': 'pie'}]])

labels = ['Same Hemi,<br>Same Net', 'Same Hemi,<br>Diff Net', 
          'Diff Hemi,<br>Same Net', 'Diff Hemi,<br>Diff Net']
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

for idx, analysis in enumerate([full_rest_analysis, full_task_analysis], 1):
    values = [analysis['within_hemi_network'], analysis['within_hemi_diff_network'],
              analysis['cross_hemi_same_network'], analysis['cross_hemi_diff_network']]
    
    fig.add_trace(go.Pie(
        labels=labels, values=values,
        marker_colors=colors,
        textinfo='percent',
        textfont_size=11,
        hole=0.3
    ), row=1, col=idx)

fig.update_layout(
    title='Full Model (232 Regions): Error Type Distribution',
    height=400, width=800,
    showlegend=True,
    legend=dict(orientation='h', y=-0.1, x=0.5, xanchor='center')
)

fig.show()

print("\nKey Observation: ~50% of errors involve CROSS-HEMISPHERE confusion (orange + red slices)")


Key Observation: ~50% of errors involve CROSS-HEMISPHERE confusion (orange + red slices)


In [11]:
# Network-level confusion matrix for Full Model
def build_major_network_cm(true_labels, pred_labels, region_info_df):
    lookup = region_info_df.set_index('region_idx')
    true_nets = pd.Series(true_labels).map(lookup['major_network'])
    pred_nets = pd.Series(pred_labels).map(lookup['major_network'])
    networks = sorted(true_nets.unique())
    cm = confusion_matrix(true_nets, pred_nets, labels=networks)
    return cm, networks

cm_full_task, networks = build_major_network_cm(task_true_labels_full, task_predictions_full, region_info_full)

# Normalize and mask diagonal
cm_norm = cm_full_task.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_norm, row_sums, where=row_sums != 0) * 100
np.fill_diagonal(cm_norm, np.nan)

fig = go.Figure(go.Heatmap(
    z=cm_norm, x=networks, y=networks,
    text=np.round(cm_norm, 1), texttemplate='%{text}',
    colorscale='Reds', showscale=True,
    colorbar=dict(title='Error %'),
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Rate: %{z:.1f}%<extra></extra>'
))

fig.update_layout(
    title='Full Model (232): Network Confusion Matrix (Task)',
    xaxis_title='Predicted Network',
    yaxis_title='True Network',
    height=500, width=600,
    yaxis=dict(autorange='reversed'),
    xaxis=dict(tickangle=45)
)

fig.show()

---
## 4. Left Hemisphere Model Analysis (116 Regions)

The left hemisphere model classifies only the 116 regions in the left hemisphere. By design, it **cannot make cross-hemisphere errors** — all errors are within-hemisphere confusions.

In [12]:
def analyze_hemisphere_model(true_labels, pred_labels, region_info_df, hemi_name, condition):
    """Analyze errors for a single-hemisphere model."""
    err_mask = true_labels != pred_labels
    total_samples = len(true_labels)
    total_errors = err_mask.sum()
    
    if total_errors == 0:
        return {'hemisphere': hemi_name, 'condition': condition, 'total_errors': 0}
    
    lookup = region_info_df.set_index('region_idx')
    
    df = pd.DataFrame({
        'true_idx': true_labels[err_mask],
        'pred_idx': pred_labels[err_mask]
    })
    
    df['true_network'] = df['true_idx'].map(lookup['network'])
    df['pred_network'] = df['pred_idx'].map(lookup['network'])
    df['true_major'] = df['true_idx'].map(lookup['major_network'])
    df['pred_major'] = df['pred_idx'].map(lookup['major_network'])
    
    same_network = df['true_network'] == df['pred_network']
    same_major = df['true_major'] == df['pred_major']
    
    # Top confusion pairs
    confusion_pairs = (df['true_major'] + ' → ' + df['pred_major']).value_counts().head(5)
    
    return {
        'hemisphere': hemi_name,
        'condition': condition,
        'total_samples': total_samples,
        'total_errors': total_errors,
        'error_rate': total_errors / total_samples,
        'within_network': same_network.sum(),
        'cross_network': (~same_network).sum(),
        'within_major': same_major.sum(),
        'cross_major': (~same_major).sum(),
        'top_confusions': confusion_pairs,
        'error_df': df
    }

# Analyze Left Hemisphere
lh_rest_analysis = analyze_hemisphere_model(cv_true_labels_lh, cv_predictions_lh, region_info_lh, 'Left', 'Rest')
lh_task_analysis = analyze_hemisphere_model(task_true_labels_lh, task_predictions_lh, region_info_lh, 'Left', 'Task')

In [13]:
# Display Left Hemisphere Analysis
print("="*85)
print("LEFT HEMISPHERE MODEL (116 REGIONS) - ERROR ANALYSIS")
print("="*85)

for analysis in [lh_rest_analysis, lh_task_analysis]:
    total = analysis['total_errors']
    print(f"\n{analysis['condition']} Condition:")
    print(f"  Total Samples: {analysis['total_samples']:,}")
    print(f"  Total Errors:  {total:,} ({analysis['error_rate']:.2%})")
    print(f"\n  Error Decomposition (ALL within Left Hemisphere):")
    print(f"  ├─ Within Same Fine Network:     {analysis['within_network']:>5,} ({analysis['within_network']/total*100:>5.1f}%)")
    print(f"  └─ Across Different Networks:    {analysis['cross_network']:>5,} ({analysis['cross_network']/total*100:>5.1f}%)")
    print(f"\n  Major Network Confusion:")
    print(f"  ├─ Within Same Major Network:    {analysis['within_major']:>5,} ({analysis['within_major']/total*100:>5.1f}%)")
    print(f"  └─ Across Major Networks:        {analysis['cross_major']:>5,} ({analysis['cross_major']/total*100:>5.1f}%)")
    print(f"\n  Top 5 Confusion Pairs:")
    for pair, count in analysis['top_confusions'].items():
        print(f"    {pair}: {count}")

print("\n" + "="*85)
print("Note: Cross-hemisphere errors are IMPOSSIBLE by design in this model.")
print("="*85)

LEFT HEMISPHERE MODEL (116 REGIONS) - ERROR ANALYSIS

Rest Condition:
  Total Samples: 25,984
  Total Errors:  2,126 (8.18%)

  Error Decomposition (ALL within Left Hemisphere):
  ├─ Within Same Fine Network:       178 (  8.4%)
  └─ Across Different Networks:    1,948 ( 91.6%)

  Major Network Confusion:
  ├─ Within Same Major Network:      699 ( 32.9%)
  └─ Across Major Networks:        1,427 ( 67.1%)

  Top 5 Confusion Pairs:
    Subcortical → Subcortical: 349
    Subcortical → Limbic: 124
    Limbic → Subcortical: 108
    Default → Default: 101
    Control → Control: 93

Task Condition:
  Total Samples: 23,200
  Total Errors:  3,032 (13.07%)

  Error Decomposition (ALL within Left Hemisphere):
  ├─ Within Same Fine Network:       236 (  7.8%)
  └─ Across Different Networks:    2,796 ( 92.2%)

  Major Network Confusion:
  ├─ Within Same Major Network:      996 ( 32.8%)
  └─ Across Major Networks:        2,036 ( 67.2%)

  Top 5 Confusion Pairs:
    Subcortical → Subcortical: 540
    S

In [14]:
# Left Hemisphere Network Confusion Matrix
cm_lh_task, networks_lh = build_major_network_cm(task_true_labels_lh, task_predictions_lh, region_info_lh)

cm_lh_norm = cm_lh_task.astype(float)
row_sums = cm_lh_norm.sum(axis=1, keepdims=True)
cm_lh_norm = np.divide(cm_lh_norm, row_sums, where=row_sums != 0) * 100
np.fill_diagonal(cm_lh_norm, np.nan)

fig = go.Figure(go.Heatmap(
    z=cm_lh_norm, x=networks_lh, y=networks_lh,
    text=np.round(cm_lh_norm, 1), texttemplate='%{text}',
    colorscale='Blues', showscale=True,
    colorbar=dict(title='Error %'),
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Rate: %{z:.1f}%<extra></extra>'
))

fig.update_layout(
    title='Left Hemisphere (116): Network Confusion Matrix (Task)',
    xaxis_title='Predicted Network',
    yaxis_title='True Network',
    height=500, width=600,
    yaxis=dict(autorange='reversed'),
    xaxis=dict(tickangle=45)
)

fig.show()

In [15]:
# Left Hemisphere: Network-level error rates
def get_network_error_rates(true_labels, pred_labels, region_info_df):
    lookup = region_info_df.set_index('region_idx')
    true_nets = pd.Series(true_labels).map(lookup['major_network'])
    correct = true_labels == pred_labels
    
    df = pd.DataFrame({'network': true_nets, 'correct': correct})
    return df.groupby('network').agg(
        total=('correct', 'count'),
        correct=('correct', 'sum')
    ).assign(error_rate=lambda x: 1 - x['correct'] / x['total'])

lh_net_rates = get_network_error_rates(task_true_labels_lh, task_predictions_lh, region_info_lh)

print("\nLeft Hemisphere: Network Error Rates (Task)")
print("-" * 50)
for net in lh_net_rates.sort_values('error_rate', ascending=False).index:
    rate = lh_net_rates.loc[net, 'error_rate']
    total = lh_net_rates.loc[net, 'total']
    print(f"  {net:30s}: {rate:>6.2%} (n={total:,})")


Left Hemisphere: Network Error Rates (Task)
--------------------------------------------------
  Subcortical                   : 31.75% (n=3,200)
  Limbic                        : 21.33% (n=1,200)
  Control                       : 11.50% (n=3,600)
  Salience/Ventral Attention    : 10.18% (n=2,200)
  Default                       :  9.96% (n=5,200)
  Somatomotor                   :  9.56% (n=3,200)
  Visual                        :  7.17% (n=2,400)
  Dorsal Attention              :  5.73% (n=2,200)


---
## 5. Right Hemisphere Model Analysis (116 Regions)

The right hemisphere model mirrors the left hemisphere analysis. Comparing the two reveals whether certain networks are more difficult to classify in one hemisphere versus the other.

In [16]:
# Analyze Right Hemisphere
rh_rest_analysis = analyze_hemisphere_model(cv_true_labels_rh, cv_predictions_rh, region_info_rh, 'Right', 'Rest')
rh_task_analysis = analyze_hemisphere_model(task_true_labels_rh, task_predictions_rh, region_info_rh, 'Right', 'Task')

In [17]:
# Display Right Hemisphere Analysis
print("="*85)
print("RIGHT HEMISPHERE MODEL (116 REGIONS) - ERROR ANALYSIS")
print("="*85)

for analysis in [rh_rest_analysis, rh_task_analysis]:
    total = analysis['total_errors']
    print(f"\n{analysis['condition']} Condition:")
    print(f"  Total Samples: {analysis['total_samples']:,}")
    print(f"  Total Errors:  {total:,} ({analysis['error_rate']:.2%})")
    print(f"\n  Error Decomposition (ALL within Right Hemisphere):")
    print(f"  ├─ Within Same Fine Network:     {analysis['within_network']:>5,} ({analysis['within_network']/total*100:>5.1f}%)")
    print(f"  └─ Across Different Networks:    {analysis['cross_network']:>5,} ({analysis['cross_network']/total*100:>5.1f}%)")
    print(f"\n  Major Network Confusion:")
    print(f"  ├─ Within Same Major Network:    {analysis['within_major']:>5,} ({analysis['within_major']/total*100:>5.1f}%)")
    print(f"  └─ Across Major Networks:        {analysis['cross_major']:>5,} ({analysis['cross_major']/total*100:>5.1f}%)")
    print(f"\n  Top 5 Confusion Pairs:")
    for pair, count in analysis['top_confusions'].items():
        print(f"    {pair}: {count}")

print("\n" + "="*85)
print("Note: Cross-hemisphere errors are IMPOSSIBLE by design in this model.")
print("="*85)

RIGHT HEMISPHERE MODEL (116 REGIONS) - ERROR ANALYSIS

Rest Condition:
  Total Samples: 25,984
  Total Errors:  2,246 (8.64%)

  Error Decomposition (ALL within Right Hemisphere):
  ├─ Within Same Fine Network:       163 (  7.3%)
  └─ Across Different Networks:    2,083 ( 92.7%)

  Major Network Confusion:
  ├─ Within Same Major Network:      687 ( 30.6%)
  └─ Across Major Networks:        1,559 ( 69.4%)

  Top 5 Confusion Pairs:
    Subcortical → Subcortical: 360
    Limbic → Subcortical: 106
    Subcortical → Limbic: 97
    Salience/Ventral Attention → Subcortical: 93
    Control → Subcortical: 89

Task Condition:
  Total Samples: 23,200
  Total Errors:  3,177 (13.69%)

  Error Decomposition (ALL within Right Hemisphere):
  ├─ Within Same Fine Network:       210 (  6.6%)
  └─ Across Different Networks:    2,967 ( 93.4%)

  Major Network Confusion:
  ├─ Within Same Major Network:      949 ( 29.9%)
  └─ Across Major Networks:        2,228 ( 70.1%)

  Top 5 Confusion Pairs:
    Subcorti

In [18]:
# Right Hemisphere Network Confusion Matrix
cm_rh_task, networks_rh = build_major_network_cm(task_true_labels_rh, task_predictions_rh, region_info_rh)

cm_rh_norm = cm_rh_task.astype(float)
row_sums = cm_rh_norm.sum(axis=1, keepdims=True)
cm_rh_norm = np.divide(cm_rh_norm, row_sums, where=row_sums != 0) * 100
np.fill_diagonal(cm_rh_norm, np.nan)

fig = go.Figure(go.Heatmap(
    z=cm_rh_norm, x=networks_rh, y=networks_rh,
    text=np.round(cm_rh_norm, 1), texttemplate='%{text}',
    colorscale='Oranges', showscale=True,
    colorbar=dict(title='Error %'),
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Rate: %{z:.1f}%<extra></extra>'
))

fig.update_layout(
    title='Right Hemisphere (116): Network Confusion Matrix (Task)',
    xaxis_title='Predicted Network',
    yaxis_title='True Network',
    height=500, width=600,
    yaxis=dict(autorange='reversed'),
    xaxis=dict(tickangle=45)
)

fig.show()

In [19]:
# Right Hemisphere: Network-level error rates
rh_net_rates = get_network_error_rates(task_true_labels_rh, task_predictions_rh, region_info_rh)

print("\nRight Hemisphere: Network Error Rates (Task)")
print("-" * 50)
for net in rh_net_rates.sort_values('error_rate', ascending=False).index:
    rate = rh_net_rates.loc[net, 'error_rate']
    total = rh_net_rates.loc[net, 'total']
    print(f"  {net:30s}: {rate:>6.2%} (n={total:,})")


Right Hemisphere: Network Error Rates (Task)
--------------------------------------------------
  Subcortical                   : 34.56% (n=3,200)
  Limbic                        : 17.62% (n=1,600)
  Salience/Ventral Attention    : 13.27% (n=3,000)
  Default                       : 11.29% (n=3,400)
  Control                       : 10.92% (n=3,800)
  Dorsal Attention              :  8.00% (n=2,200)
  Somatomotor                   :  7.47% (n=3,600)
  Visual                        :  6.13% (n=2,400)


---
## 6. Cross-Strategy Comparison

### What Does Hemisphere Separation Reveal?

By comparing the Full model to the hemisphere-specific models, we can quantify:
1. How much error is due to **hemisphere confusion**
2. How the **left and right hemispheres differ** in classification difficulty
3. Which **networks are problematic** regardless of hemisphere

In [20]:
# Comprehensive comparison table
comparison_data = []

for analysis, model in [(full_rest_analysis, 'Full (232)'), (full_task_analysis, 'Full (232)'),
                         (lh_rest_analysis, 'Left (116)'), (lh_task_analysis, 'Left (116)'),
                         (rh_rest_analysis, 'Right (116)'), (rh_task_analysis, 'Right (116)')]:
    
    total = analysis['total_errors']
    
    # For hemisphere models, cross-hemi errors are 0
    if 'cross_hemi_same_network' in analysis:
        cross_hemi = analysis['cross_hemi_same_network'] + analysis['cross_hemi_diff_network']
        cross_hemi_pct = cross_hemi / total * 100 if total > 0 else 0
    else:
        cross_hemi = 0
        cross_hemi_pct = 0
    
    comparison_data.append({
        'Model': model,
        'Condition': analysis['condition'],
        'Total_Errors': total,
        'Error_Rate': analysis['error_rate'] * 100,
        'Cross_Hemi_Errors': cross_hemi,
        'Cross_Hemi_Pct': cross_hemi_pct
    })

comparison_df = pd.DataFrame(comparison_data)

print("="*90)
print("CROSS-STRATEGY COMPARISON")
print("="*90)
print(comparison_df.to_string(index=False))
print("="*90)

CROSS-STRATEGY COMPARISON
      Model Condition  Total_Errors  Error_Rate  Cross_Hemi_Errors  Cross_Hemi_Pct
 Full (232)      Rest          3943    7.587361               2032       51.534365
 Full (232)      Task          4993   10.760776               2457       49.208892
 Left (116)      Rest          2126    8.181958                  0        0.000000
 Left (116)      Task          3032   13.068966                  0        0.000000
Right (116)      Rest          2246    8.643781                  0        0.000000
Right (116)      Task          3177   13.693966                  0        0.000000


In [21]:
# Left vs Right Hemisphere Comparison
print("="*85)
print("LEFT vs RIGHT HEMISPHERE COMPARISON (Task Condition)")
print("="*85)

print(f"\n{'Metric':<35} {'Left Hemisphere':<20} {'Right Hemisphere':<20}")
print("-" * 75)
print(f"{'Total Errors':<35} {lh_task_analysis['total_errors']:<20,} {rh_task_analysis['total_errors']:<20,}")
print(f"{'Error Rate':<35} {lh_task_analysis['error_rate']*100:<20.2f}% {rh_task_analysis['error_rate']*100:<20.2f}%")
print(f"{'Within-Network Errors':<35} {lh_task_analysis['within_network']:<20,} {rh_task_analysis['within_network']:<20,}")
print(f"{'Cross-Network Errors':<35} {lh_task_analysis['cross_network']:<20,} {rh_task_analysis['cross_network']:<20,}")

# Hemisphere asymmetry
lh_rate = lh_task_analysis['error_rate'] * 100
rh_rate = rh_task_analysis['error_rate'] * 100
diff = rh_rate - lh_rate

print(f"\n{'Hemisphere Asymmetry:':<35} {abs(diff):.2f}% {'higher in Right' if diff > 0 else 'higher in Left'}")
print("="*85)

LEFT vs RIGHT HEMISPHERE COMPARISON (Task Condition)

Metric                              Left Hemisphere      Right Hemisphere    
---------------------------------------------------------------------------
Total Errors                        3,032                3,177               
Error Rate                          13.07               % 13.69               %
Within-Network Errors               236                  210                 
Cross-Network Errors                2,796                2,967               

Hemisphere Asymmetry:               0.63% higher in Right


In [22]:
# Network-by-network comparison between hemispheres
lh_rates = get_network_error_rates(task_true_labels_lh, task_predictions_lh, region_info_lh)
rh_rates = get_network_error_rates(task_true_labels_rh, task_predictions_rh, region_info_rh)

network_comparison = pd.DataFrame({
    'Network': lh_rates.index,
    'LH_Error_Rate': lh_rates['error_rate'].values * 100,
    'RH_Error_Rate': rh_rates['error_rate'].values * 100
}).assign(
    Difference=lambda x: x['RH_Error_Rate'] - x['LH_Error_Rate'],
    Harder_In=lambda x: np.where(x['Difference'] > 0, 'Right', 'Left')
)

print("\nNetwork Error Rates: Left vs Right Hemisphere (Task)")
print("="*70)
print(network_comparison.to_string(index=False))
print("\nPositive Difference = Higher error rate in Right Hemisphere")


Network Error Rates: Left vs Right Hemisphere (Task)
                   Network  LH_Error_Rate  RH_Error_Rate  Difference Harder_In
                   Control      11.500000      10.921053   -0.578947      Left
                   Default       9.961538      11.294118    1.332579     Right
          Dorsal Attention       5.727273       8.000000    2.272727     Right
                    Limbic      21.333333      17.625000   -3.708333      Left
Salience/Ventral Attention      10.181818      13.266667    3.084848     Right
               Somatomotor       9.562500       7.472222   -2.090278      Left
               Subcortical      31.750000      34.562500    2.812500     Right
                    Visual       7.166667       6.125000   -1.041667      Left

Positive Difference = Higher error rate in Right Hemisphere


In [23]:
# Visualize hemisphere comparison
fig = go.Figure()

networks_list = network_comparison['Network'].tolist()

fig.add_trace(go.Bar(
    name='Left Hemisphere',
    x=networks_list,
    y=network_comparison['LH_Error_Rate'],
    marker_color='#3498db'
))

fig.add_trace(go.Bar(
    name='Right Hemisphere',
    x=networks_list,
    y=network_comparison['RH_Error_Rate'],
    marker_color='#e67e22'
))

fig.update_layout(
    title='Network Error Rates: Left vs Right Hemisphere (Task)',
    xaxis_title='Network',
    yaxis_title='Error Rate (%)',
    barmode='group',
    template='plotly_white',
    height=450, width=900,
    xaxis_tickangle=45
)

fig.show()

In [24]:
# Quantify hemisphere contribution to Full model errors
print("="*85)
print("HEMISPHERE CONFUSION CONTRIBUTION TO FULL MODEL ERRORS")
print("="*85)

for condition, full_analysis in [('Rest', full_rest_analysis), ('Task', full_task_analysis)]:
    total = full_analysis['total_errors']
    cross_hemi = full_analysis['cross_hemi_same_network'] + full_analysis['cross_hemi_diff_network']
    within_hemi = full_analysis['within_hemi_network'] + full_analysis['within_hemi_diff_network']
    
    print(f"\n{condition} Condition:")
    print(f"  Total Full Model Errors:       {total:,}")
    print(f"  ├─ Within-Hemisphere Errors:   {within_hemi:,} ({within_hemi/total*100:.1f}%)")
    print(f"  └─ Cross-Hemisphere Errors:    {cross_hemi:,} ({cross_hemi/total*100:.1f}%)")
    print(f"")
    print(f"  → {cross_hemi/total*100:.1f}% of Full model errors are ELIMINATED")
    print(f"    by using hemisphere-specific models.")

print("\n" + "="*85)

HEMISPHERE CONFUSION CONTRIBUTION TO FULL MODEL ERRORS

Rest Condition:
  Total Full Model Errors:       3,943
  ├─ Within-Hemisphere Errors:   1,911 (48.5%)
  └─ Cross-Hemisphere Errors:    2,032 (51.5%)

  → 51.5% of Full model errors are ELIMINATED
    by using hemisphere-specific models.

Task Condition:
  Total Full Model Errors:       4,993
  ├─ Within-Hemisphere Errors:   2,536 (50.8%)
  └─ Cross-Hemisphere Errors:    2,457 (49.2%)

  → 49.2% of Full model errors are ELIMINATED
    by using hemisphere-specific models.



---
## 7. Limitations of Multinomial Classification

Despite separating hemispheres, multinomial logistic regression has fundamental limitations that prevent us from answering key research questions.

In [25]:
print("="*90)
print("LIMITATIONS OF MULTINOMIAL LOGISTIC REGRESSION")
print("="*90)

print("""
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                     WHAT MULTINOMIAL CANNOT TELL US                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  1. WHICH SPECIFIC REGION PAIRS ARE CONFUSED?                                       │
│     • Multinomial gives a single softmax over all classes                           │
│     • Cannot isolate pairwise discriminability                                      │
│     • Cannot answer: "Is Region A vs B harder than Region A vs C?"                  │
│                                                                                     │
│  2. WHAT MAKES A REGION DIFFICULT TO CLASSIFY?                                      │
│     • All regions share the same decision boundary structure                        │
│     • Cannot identify region-specific classification challenges                     │
│     • Cannot answer: "Why is Subcortical harder than Visual?"                       │
│                                                                                     │
│  3. HOW DOES TASK STATE AFFECT SPECIFIC REGION PAIRS?                               │
│     • Global accuracy drop doesn't reveal selective effects                         │
│     • Cannot answer: "Which region pairs become MORE confusable during task?"       │
│     • Cannot identify task-specific functional reorganization                       │
│                                                                                     │
│  4. CAN WE OPTIMIZE CLASSIFICATION PER REGION?                                      │
│     • Single threshold for all classes                                              │
│     • No per-region confidence calibration                                          │
│     • Cannot tune sensitivity/specificity per region                                │
│                                                                                     │
│  5. WHAT IS THE DISCRIMINABILITY OF EACH REGION'S FINGERPRINT?                      │
│     • Multinomial conflates all sources of error                                    │
│     • Cannot measure "fingerprint uniqueness" per region                            │
│     • Cannot rank regions by classification difficulty                              │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
""")

print("="*90)

LIMITATIONS OF MULTINOMIAL LOGISTIC REGRESSION

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                     WHAT MULTINOMIAL CANNOT TELL US                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  1. WHICH SPECIFIC REGION PAIRS ARE CONFUSED?                                       │
│     • Multinomial gives a single softmax over all classes                           │
│     • Cannot isolate pairwise discriminability                                      │
│     • Cannot answer: "Is Region A vs B harder than Region A vs C?"                  │
│                                                                                     │
│  2. WHAT MAKES A REGION DIFFICULT TO CLASSIFY?                                      │
│     • All regions share the same decision boundary structure          

In [26]:
# Demonstrate limitation: We can see WHAT is confused but not WHY
print("\nExample: Top Confusions in Left Hemisphere (Task)")
print("-" * 60)

# Get detailed confusion pairs
err_df = lh_task_analysis['error_df']
region_pairs = (err_df['true_idx'].astype(str) + ' → ' + err_df['pred_idx'].astype(str)).value_counts().head(10)

lookup = region_info_lh.set_index('region_idx')

print(f"\n{'True Region':<30} {'Predicted Region':<30} {'Count':>6}")
print("="*70)

for pair, count in region_pairs.items():
    true_idx, pred_idx = map(int, pair.split(' → '))
    true_name = lookup.loc[true_idx, 'region_name'] if true_idx in lookup.index else f'Region {true_idx}'
    pred_name = lookup.loc[pred_idx, 'region_name'] if pred_idx in lookup.index else f'Region {pred_idx}'
    print(f"{true_name:<30} {pred_name:<30} {count:>6}")

print("\n" + "-"*70)
print("LIMITATION: We see these pairs are confused, but multinomial cannot tell us:")
print("  • How difficult is THIS specific pair vs other pairs?")
print("  • What features drive this confusion?")
print("  • How does task state change THIS pair's discriminability?")


Example: Top Confusions in Left Hemisphere (Task)
------------------------------------------------------------

True Region                    Predicted Region                Count
THA-VA-lh                      aGP-lh                             21
pGP-lh                         THA-DA-lh                          16
aGP-lh                         pPUT-lh                            13
THA-DA-lh                      pGP-lh                             11
THA-DA-lh                      aGP-lh                             10
pCAU-lh                        aGP-lh                             10
LH_SomMotB_S2_1                THA-VA-lh                           9
LH_DefaultB_PFCv_2             LH_LimbicA_TempPole_3               9
LH_VisPeri_ExStrSup_1          LH_VisPeri_ExStrInf_3               9
aGP-lh                         THA-DP-lh                           9

----------------------------------------------------------------------
LIMITATION: We see these pairs are confused, but multino

---
## 8. Motivation for One-vs-Rest (OvR) and One-vs-One (OvO)

To answer the questions that multinomial cannot address, we need decomposed classification strategies.

In [27]:
print("="*90)
print("WHY WE NEED ONE-VS-REST (OvR) AND ONE-VS-ONE (OvO)")
print("="*90)

print("""
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                           ONE-VS-REST (OvR) STRATEGY                                │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  APPROACH: Train K binary classifiers, each distinguishing one region from all     │
│            other regions combined.                                                  │
│                                                                                     │
│  For 116 regions → 116 binary classifiers                                           │
│                                                                                     │
│  ANSWERS PROVIDED:                                                                  │
│                                                                                     │
│  ✓ Per-region discriminability                                                      │
│    → "How distinguishable is Region A from ALL other regions?"                      │
│    → Measures the uniqueness of each region's connectivity fingerprint              │
│                                                                                     │
│  ✓ Region-specific sensitivity/specificity                                          │
│    → Can optimize thresholds per region                                             │
│    → Identify regions with high false positive vs false negative rates              │
│                                                                                     │
│  ✓ Task-induced changes per region                                                  │
│    → "Does Region A become harder to identify during task?"                         │
│    → Measures functional reorganization at the region level                         │
│                                                                                     │
│  ✓ Feature importance per region                                                    │
│    → Which connections are most diagnostic for each region?                         │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
""")

WHY WE NEED ONE-VS-REST (OvR) AND ONE-VS-ONE (OvO)

┌─────────────────────────────────────────────────────────────────────────────────────┐
│                           ONE-VS-REST (OvR) STRATEGY                                │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  APPROACH: Train K binary classifiers, each distinguishing one region from all     │
│            other regions combined.                                                  │
│                                                                                     │
│  For 116 regions → 116 binary classifiers                                           │
│                                                                                     │
│  ANSWERS PROVIDED:                                                                  │
│                                                                    

In [28]:
print("""
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                           ONE-VS-ONE (OvO) STRATEGY                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  APPROACH: Train K(K-1)/2 binary classifiers, each distinguishing one specific     │
│            pair of regions.                                                         │
│                                                                                     │
│  For 116 regions → 6,670 pairwise classifiers                                       │
│                                                                                     │
│  ANSWERS PROVIDED:                                                                  │
│                                                                                     │
│  ✓ Pairwise discriminability matrix                                                 │
│    → "How hard is it to distinguish Region A from Region B specifically?"           │
│    → Creates a 116×116 discriminability matrix                                      │
│                                                                                     │
│  ✓ Identification of confusable pairs                                               │
│    → Which specific region pairs have overlapping fingerprints?                     │
│    → Quantifies similarity at the pair level                                        │
│                                                                                     │
│  ✓ Task-specific pairwise changes                                                   │
│    → "Does the A-vs-B distinction become harder during task?"                       │
│    → Reveals which specific relationships change under cognitive load               │
│                                                                                     │
│  ✓ Network-level discriminability patterns                                          │
│    → Are within-network pairs harder than between-network pairs?                    │
│    → Quantifies functional organization at the pairwise level                       │
│                                                                                     │
│  ✓ Error-as-signal analysis                                                         │
│    → Pairs that become MORE confusable during task indicate                         │
│      functional reorganization (the core thesis hypothesis)                         │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
""")


┌─────────────────────────────────────────────────────────────────────────────────────┐
│                           ONE-VS-ONE (OvO) STRATEGY                                 │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  APPROACH: Train K(K-1)/2 binary classifiers, each distinguishing one specific     │
│            pair of regions.                                                         │
│                                                                                     │
│  For 116 regions → 6,670 pairwise classifiers                                       │
│                                                                                     │
│  ANSWERS PROVIDED:                                                                  │
│                                                                                     │
│  ✓ Pairwise discriminability m

In [29]:
print("""
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    RESEARCH QUESTIONS ADDRESSED BY EACH APPROACH                    │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  QUESTION                                          MULTI   OvR    OvO               │
│  ─────────────────────────────────────────────────────────────────────────          │
│  Overall classification accuracy                    ✓       ✓      ✓                │
│  Network-level error patterns                       ✓       ✓      ✓                │
│  Per-region discriminability                        ✗       ✓      ○                │
│  Pairwise discriminability                          ✗       ✗      ✓                │
│  Region-specific threshold optimization             ✗       ✓      ✗                │
│  Task-induced changes per region                    ○       ✓      ✓                │
│  Task-induced changes per pair                      ✗       ✗      ✓                │
│  Feature importance per region                      ✗       ✓      ○                │
│  Identification of confusable pairs                 ○       ✗      ✓                │
│  Fingerprint uniqueness ranking                     ✗       ✓      ✓                │
│                                                                                     │
│  Legend: ✓ = Directly answers  ○ = Partially answers  ✗ = Cannot answer             │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
""")

print("="*90)


┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    RESEARCH QUESTIONS ADDRESSED BY EACH APPROACH                    │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│  QUESTION                                          MULTI   OvR    OvO               │
│  ─────────────────────────────────────────────────────────────────────────          │
│  Overall classification accuracy                    ✓       ✓      ✓                │
│  Network-level error patterns                       ✓       ✓      ✓                │
│  Per-region discriminability                        ✗       ✓      ○                │
│  Pairwise discriminability                          ✗       ✗      ✓                │
│  Region-specific threshold optimization             ✗       ✓      ✗                │
│  Task-induced changes per reg

In [30]:
# Summary statistics
print("="*90)
print("SUMMARY: MULTINOMIAL ANALYSIS FINDINGS")
print("="*90)

print(f"""
MODEL PERFORMANCE (Task Condition):
  • Full (232 regions):    {task_summary_full['task_test_accuracy']:.2%} accuracy
  • Left Hemisphere:       {task_summary_lh['task_test_accuracy']:.2%} accuracy  
  • Right Hemisphere:      {task_summary_rh['task_test_accuracy']:.2%} accuracy

ERROR DECOMPOSITION (Full Model - Task):
  • Cross-hemisphere errors: {full_task_analysis['cross_hemi_same_network'] + full_task_analysis['cross_hemi_diff_network']:,} ({(full_task_analysis['cross_hemi_same_network'] + full_task_analysis['cross_hemi_diff_network'])/full_task_analysis['total_errors']*100:.1f}%)
  • Within-hemisphere errors: {full_task_analysis['within_hemi_network'] + full_task_analysis['within_hemi_diff_network']:,} ({(full_task_analysis['within_hemi_network'] + full_task_analysis['within_hemi_diff_network'])/full_task_analysis['total_errors']*100:.1f}%)

KEY FINDINGS:
  1. ~50% of Full model errors are due to hemisphere confusion
  2. Hemisphere-specific models eliminate this error source entirely
  3. Remaining errors are network confusions (especially Subcortical)
  4. Left and Right hemispheres show slightly different error patterns

LIMITATIONS IDENTIFIED:
  • Cannot measure per-region discriminability
  • Cannot identify specific confusable pairs
  • Cannot quantify task-induced changes at region/pair level
  • Cannot optimize classification per region

NEXT STEPS:
  → Implement One-vs-Rest (OvR) for per-region analysis
  → Implement One-vs-One (OvO) for pairwise discriminability
  → Apply to both hemispheres separately
  → Compare rest vs task at the region and pair level
""")

print("="*90)

SUMMARY: MULTINOMIAL ANALYSIS FINDINGS

MODEL PERFORMANCE (Task Condition):
  • Full (232 regions):    89.24% accuracy
  • Left Hemisphere:       86.93% accuracy  
  • Right Hemisphere:      86.31% accuracy

ERROR DECOMPOSITION (Full Model - Task):
  • Cross-hemisphere errors: 2,457 (49.2%)
  • Within-hemisphere errors: 2,536 (50.8%)

KEY FINDINGS:
  1. ~50% of Full model errors are due to hemisphere confusion
  2. Hemisphere-specific models eliminate this error source entirely
  3. Remaining errors are network confusions (especially Subcortical)
  4. Left and Right hemispheres show slightly different error patterns

LIMITATIONS IDENTIFIED:
  • Cannot measure per-region discriminability
  • Cannot identify specific confusable pairs
  • Cannot quantify task-induced changes at region/pair level
  • Cannot optimize classification per region

NEXT STEPS:
  → Implement One-vs-Rest (OvR) for per-region analysis
  → Implement One-vs-One (OvO) for pairwise discriminability
  → Apply to both he